In [3]:
from datasets import load_dataset
import pandas as pd

# Загружаем датасет с полным именем
dataset = load_dataset("fancyzhx/ag_news")

train = pd.DataFrame(dataset['train'])
test = pd.DataFrame(dataset['test'])

# Переименовываем колонки
train = train.rename(columns={'text': 'text', 'label': 'label'})
test = test.rename(columns={'text': 'text', 'label': 'label'})

# Ограничим размер для скорости (2000 train, 300 test)
train = train.sample(2000, random_state=42)
test = test.sample(300, random_state=42)

# Сохраняем тест
test.to_csv('test.csv', index=False)
print(" Данные загружены! Обучающая:", len(train), "Тестовая:", len(test))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Данные загружены! Обучающая: 2000 Тестовая: 300


In [4]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("=== ПОДХОД А: TF-IDF + Логистическая регрессия ===")

# Создаем пайплайн
pipe = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, analyzer='word'),
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
)

t0 = time.perf_counter()
pipe.fit(train['text'], train['label'])
train_time = time.perf_counter() - t0
print(f"Время обучения: {train_time:.2f} сек")

t0 = time.perf_counter()
pred_a = pipe.predict(test['text'])
pred_time = (time.perf_counter() - t0) / len(test) * 1000
print(f"Время предсказания 1000 текстов: {pred_time:.2f} сек")

# Метрики
macro_f1_a = f1_score(test['label'], pred_a, average='macro')
acc_a = accuracy_score(test['label'], pred_a)

print(f"\n--- РЕЗУЛЬТАТЫ ПОДХОДА А ---")
print(f"Macro-F1: {macro_f1_a:.4f}")
print(f"Accuracy: {acc_a:.4f}")
print("\nОтчет по классам:")
print(classification_report(test['label'], pred_a, zero_division=0))

=== ПОДХОД А: TF-IDF + Логистическая регрессия ===
Время обучения: 4.26 сек
Время предсказания 1000 текстов: 0.52 сек

--- РЕЗУЛЬТАТЫ ПОДХОДА А ---
Macro-F1: 0.8310
Accuracy: 0.8333

Отчет по классам:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84        71
           1       0.86      0.93      0.89        80
           2       0.84      0.76      0.80        71
           3       0.82      0.77      0.79        78

    accuracy                           0.83       300
   macro avg       0.83      0.83      0.83       300
weighted avg       0.83      0.83      0.83       300



In [5]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import time

print("=== ПОДХОД В: Дообучение трансформера ===")

# 1. Загружаем компактную модель
MODEL = 'cointegrated/rubert-tiny2'
labels = sorted(train['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))

# 2. Функция токенизации
def encode(batch):
    out = tok(batch['text'], truncation=True, max_length=256, padding='max_length')
    out['labels'] = [label2id[l] for l in batch['label']]
    return out

ds_train = Dataset.from_pandas(train).map(encode, batched=True)
ds_test = Dataset.from_pandas(test).map(encode, batched=True)

# 3. Метрики для оценки
def compute_metrics(eval_pred):
    logits, y = eval_pred
    return {'macro_f1': f1_score(y, logits.argmax(-1), average='macro')}

# 4. Настройки обучения
args = TrainingArguments(
    output_dir='./bert_out', num_train_epochs=3, # 3 эпохи для скорости
    per_device_train_batch_size=16, learning_rate=3e-5,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    seed=42, report_to=[], logging_steps=100
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_test, compute_metrics=compute_metrics)

# 5. Обучение
t0 = time.perf_counter()
trainer.train()
train_time_b = (time.perf_counter() - t0) / 60
print(f"\nВремя обучения: {train_time_b:.2f} мин")

# 6. Оценка на тесте
pred_b = trainer.predict(ds_test).predictions.argmax(-1)
macro_f1_b = f1_score([label2id[l] for l in test['label']], pred_b, average='macro')
acc_b = accuracy_score([label2id[l] for l in test['label']], pred_b)

print(f"\n--- РЕЗУЛЬТАТЫ ПОДХОДА В (дообучение) ---")
print(f"Macro-F1: {macro_f1_b:.4f}")
print(f"Accuracy: {acc_b:.4f}")

=== ПОДХОД В: Дообучение трансформера ===


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Время обучения: 0.80 мин



--- РЕЗУЛЬТАТЫ ПОДХОДА В (дообучение) ---
Macro-F1: 0.8119
Accuracy: 0.8167


In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

print("=== ПОДХОД В (Эмбеддинги) ===")

# Используем компактную модель для эмбеддингов
encoder = SentenceTransformer('intfloat/multilingual-e5-small', device='cuda')

# Получаем эмбеддинги
emb_train = encoder.encode(list(train['text']), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb_test = encoder.encode(list(test['text']), batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# Обучаем логистическую регрессию поверх эмбеддингов
clf = LogisticRegression(max_iter=3000, class_weight='balanced')
clf.fit(emb_train, train['label'])
pred_b2 = clf.predict(emb_test)

macro_f1_b2 = f1_score(test['label'], pred_b2, average='macro')
acc_b2 = accuracy_score(test['label'], pred_b2)

print(f"\n--- РЕЗУЛЬТАТЫ ПОДХОДА В (эмбеддинги) ---")
print(f"Macro-F1: {macro_f1_b2:.4f}")
print(f"Accuracy: {acc_b2:.4f}")

=== ПОДХОД В (Эмбеддинги) ===


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (8 outputs left out)

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


--- РЕЗУЛЬТАТЫ ПОДХОДА В (эмбеддинги) ---
Macro-F1: 0.8743
Accuracy: 0.8767


In [7]:
!pip install -q transformers accelerate bitsandbytes

import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("=== ПОДХОД С: LLM (Qwen2.5-3B) ===")

# Загружаем модель с 4-битной квантизацией
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
LLM = 'Qwen/Qwen2.5-3B-Instruct'

llm_tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM, quantization_config=bnb, device_map='auto')

# Названия классов для промпта (0, 1, 2, 3 — это классы AG News)
label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
labels = list(label_names.values())

SYSTEM_PROMPT = (
    "Ты классифицируешь новости. Допустимые классы: World, Sports, Business, Sci/Tech. "
    "Ответь строго одним объектом JSON вида {\"label\": \"<класс>\"}. "
    "Никакого текста вне JSON."
)

def parse_answer(raw):
    m = re.search(r'\{.*}', raw, flags=re.S)
    if not m: return "World", "no JSON" # Запасной класс
    try:
        obj = json.loads(m.group(0))
        if obj.get('label') in labels:
            return obj['label'], None
        else:
            return "World", "class out of list"
    except:
        return "World", "bad JSON"

def classify(text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text[:1000]}]
    prompt = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(prompt, return_tensors='pt').to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=32, do_sample=False, pad_token_id=llm_tok.eos_token_id)
    raw = llm_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return parse_answer(raw)

# Прогоняем первые 50 текстов
test_subset = test.head(50)
pred_c, bad = [], 0
for t in test_subset['text']:
    label, err = classify(t)
    if err: bad += 1
    # Переводим текстовый лейбл обратно в число
    pred_c.append([k for k, v in label_names.items() if v == label][0])

from sklearn.metrics import f1_score, accuracy_score
macro_f1_c = f1_score(test_subset['label'], pred_c, average='macro')
acc_c = accuracy_score(test_subset['label'], pred_c)

print(f"\n--- РЕЗУЛЬТАТЫ ПОДХОДА С (LLM на 50 текстах) ---")
print(f"Macro-F1: {macro_f1_c:.4f}")
print(f"Accuracy: {acc_c:.4f}")
print(f"Доля ответов вне схемы JSON: {bad / len(test_subset) * 100:.2f}%")

=== ПОДХОД С: LLM (Qwen2.5-3B) ===


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


--- РЕЗУЛЬТАТЫ ПОДХОДА С (LLM на 50 текстах) ---
Macro-F1: 0.7120
Accuracy: 0.7800
Доля ответов вне схемы JSON: 4.00%
